In [92]:
import re
from typing import List, Dict
from typing import Dict, List
from datetime import datetime

# Departamentos validos para empleados
DEPARTAMENTOS_VALIDOS = ['VEN', 'ADM', 'TEC', 'LOG', 'RHH']

# Series validas para facturas
SERIES_VALIDAS = ['A', 'B', 'C', 'D', 'E']

In [93]:
# PARTE 1: Funciones de validacion individual
def validar_producto(codigo: str) -> Dict:
  resultado = {"valido": False, "categoria": None, "numero": None, "pais": None}
  patron = r'^([A-Z]{3})-(\d{4})-([A-Z]{2})$'
  m = re.match(patron, codigo)
  if m:
    resultado.update({
        "valido": True,
        "categoria": m.group(1),
        "numero": m.group(2),
        "pais": m.group(3)
    })
  return resultado

def validar_envio(codigo: str) -> Dict:
  resultado={"valido": False, "fecha": None, "secuencial": None}
  patron = r'^ENV-(\d{4})-(\d{2})-(\d{2})-(\d{6})$'
  m = re.match(patron, codigo)
  if m:
    año, mes, dia, sec = m.group(1), m.group(2), m.group(3), m.group(4)
    try:
      if 2020 <= int(año) <= 2023 and 1 <= int(mes) <= 12 and 1 <= int(dia) <= 31:
        datetime.date(int(año), int(mes), int(dia))
        resultado.update({
            "valido": True,
            "fecha": f"{int(año)}-{int(mes):02d}-{int(dia):02d}",
            "secuencial": sec
        })
    except ValueError:
      pass
  return resultado

def validar_empleado(codigo: str) -> Dict:
  resultado = {"valido": False, "departamento": None, "numero": None}
  patron = r'^([A-Z]{3})-(\d{4})$'
  m = re.match(patron, codigo)
  if m:
    dept, num = m.group(1), m.group(2)
    if dept in DEPARTAMENTOS_VALIDOS and not num.startswith('0'):
      resultado.update({"valido": True, "departamento": dept,"numero": num})
  return resultado

def validar_factura(codigo: str) -> Dict:
  resultado = {"valido": False, "serie": None, "numero": None}
  patron = r'^FAC-([A-Z])-([0-9]{6})$'
  m = re.match(patron, codigo)
  if m:
    serie, numero = m.group(1), m.group(2)
    if serie in SERIES_VALIDAS:
      resultado.update({"valido": True, "serie": serie, "numero": numero})
  return resultado

In [94]:
# PARTE 2: Validador Universal

def validar_codigo(codigo: str) -> Dict:
  resultado = {"codigo": codigo, "tipo": None, "valido": False, "detalles": {}}
  if re.match(r'^[A-Z]{3}-\d{4}-\w{2}$', codigo):
    resultado['tipo'] = 'producto'
    resultado['detalles'] = validar_producto(codigo)
    resultado['valido'] = resultado['detalles']['valido']
  elif codigo.startswith('ENV-'):
    resultado['tipo'] = 'envio'
    resultado['detalles'] = validar_envio(codigo)
    resultado['valido'] = resultado['detalles']['valido']
  elif codigo.startswith('EMP-'):
    resultado['tipo'] = 'empleado'
    resultado['detalles'] = validar_empleado(codigo)
    resultado['valido'] = resultado['detalles']['valido']
  elif codigo.startswith('FAC-'):
    resultado['tipo'] = 'factura'
    resultado['detalles'] = validar_factura(codigo)
    resultado['valido'] = resultado['detalles']['valido']
  return resultado

In [95]:
# PARTE 3: Procesamiento de lotes

def procesar_lote(lineas: List[str]) -> Dict:
  resultado = {"total": 0, "validos": 0, "invalidos": 0, "por_tipo": {
       "producto": {"total":0, "validos":0},
        "envio": {"total":0, "validos":0},
        "empleado": {"total":0, "validos":0},
        "factura": {"total":0, "validos":0},
        "desconocido": {"total":0, "validos":0}
    }, "detalle": []}

  for cod in lineas: # Changed 'codigos' to 'lineas'
    res = validar_codigo(cod)
    resultado['total'] += 1
    if res['valido']:
      resultado['validos'] += 1
    else:
      resultado['invalidos'] += 1
    tipo = res['tipo']
    if tipo not in resultado['por_tipo']:
      tipo = 'desconocido'
    resultado['por_tipo'][tipo]['total'] += 1
    if res['valido']:
      resultado['por_tipo'][tipo]['validos'] += 1
    resultado['detalle'].append(res)
  return resultado

In [96]:
# Codigo de prueba
CODIGOS_PRUEBA = [
    # Productos
    "TEC-0001-MX",      # Válido
    "ALI-9999-US",      # Válido
    "ROB-1234-CA",      # Válido
    "tec-0001-MX",      # Inválido: minúsculas
    "TEC-001-MX",       # Inválido: solo 3 dígitos
    "TECH-0001-MX",     # Inválido: 4 letras en categoría

    # Envíos
    "ENV-2024-03-15-001234",    # Válido
    "ENV-2025-12-01-999999",    # Válido
    "ENV-2019-03-15-001234",    # Inválido: año fuera de rango
    "ENV-2024-13-15-001234",    # Inválido: mes 13
    "ENV-2024-03-32-001234",    # Inválido: día 32

    # Empleados
    "EMP-VEN-1234",     # Válido
    "EMP-TEC-9999",     # Válido
    "EMP-ADM-1000",     # Válido
    "EMP-VEN-0123",     # Inválido: empieza con 0
    "EMP-XXX-1234",     # Inválido: departamento no válido
    "EMP-VEN-123",      # Inválido: solo 3 dígitos

    # Facturas
    "FAC-A-123456",     # Válido
    "FAC-E-000001",     # Válido
    "FAC-B-999999",     # Válido
    "FAC-F-123456",     # Inválido: serie F no existe
    "FAC-A-12345",      # Inválido: solo 5 dígitos
    "FAC-a-123456",     # Inválido: serie en minúscula

    # Desconocidos
    "XXX-1234",         # Desconocido
    "RANDOM-CODE",      # Desconocido
]

In [97]:
#Funcion para mostrar resultados

def mostrar_resultado(resultado: Dict) -> None:
    estado = "✓" if resultado["valido"] else "✗"
    print(f"{estado} {resultado['codigo']:<30} | Tipo: {resultado['tipo']:<12}")
    if resultado["valido"] and resultado["detalles"]:
        detalles = ", ".join(f"{k}: {v}" for k, v in resultado["detalles"].items() if v)
        print(f"   └── {detalles}")

def mostrar_reporte(reporte: Dict) -> None:
    print("=" * 60)
    print("                 REPORTE DE VALIDACIÓN")
    print("=" * 60)
    print(f"\nTotal procesados: {reporte['total']}")
    print(f"Válidos: {reporte['validos']} ({reporte['validos']/reporte['total']*100:.1f}%)")
    print(f"Inválidos: {reporte['invalidos']} ({reporte['invalidos']/reporte['total']*100:.1f}%)")

    print("\nDesglose por tipo:")
    print("-" * 40)
    for tipo, stats in reporte["por_tipo"].items():
        if stats["total"] > 0:
            tasa = stats["validos"] / stats["total"] * 100
            print(f"  {tipo.capitalize():<12}: {stats['validos']:>3}/{stats['total']:<3} ({tasa:.0f}% válidos)")

    print("\n" + "=" * 60)

In [98]:
#Pruebas validacion individual

print("PRUEBA DE FUNCIONES INDIVIDUALES")
print("=" * 50)

# Producto
print("\n-- Productos --")
prod_code1 = "TEC-0001-MX"
result_prod1 = validar_producto(prod_code1)
mostrar_resultado({
    "codigo": prod_code1,
    "tipo": "producto",
    "valido": result_prod1["valido"],
    "detalles": result_prod1
})
prod_code2 = "tec-0001-MX"
result_prod2 = validar_producto(prod_code2)
mostrar_resultado({
    "codigo": prod_code2,
    "tipo": "producto",
    "valido": result_prod2["valido"],
    "detalles": result_prod2
})

# Envío
print("\n-- Envíos --")
envio_code1 = "ENV-2024-03-15-001234"
result_envio1 = validar_envio(envio_code1)
mostrar_resultado({
    "codigo": envio_code1,
    "tipo": "envio",
    "valido": result_envio1["valido"],
    "detalles": result_envio1
})
envio_code2 = "ENV-2024-13-15-001234"
result_envio2 = validar_envio(envio_code2)
mostrar_resultado({
    "codigo": envio_code2,
    "tipo": "envio",
    "valido": result_envio2["valido"],
    "detalles": result_envio2
})

# Empleado
print("\n-- Empleados --")
emp_code1 = "EMP-VEN-1234"
result_emp1 = validar_empleado(emp_code1)
mostrar_resultado({
    "codigo": emp_code1,
    "tipo": "empleado",
    "valido": result_emp1["valido"],
    "detalles": result_emp1
})
emp_code2 = "EMP-VEN-0123"
result_emp2 = validar_empleado(emp_code2)
mostrar_resultado({
    "codigo": emp_code2,
    "tipo": "empleado",
    "valido": result_emp2["valido"],
    "detalles": result_emp2
})

# Factura
print("\n-- Facturas --")
fac_code1 = "FAC-A-123456"
result_fac1 = validar_factura(fac_code1)
mostrar_resultado({
    "codigo": fac_code1,
    "tipo": "factura",
    "valido": result_fac1["valido"],
    "detalles": result_fac1
})
fac_code2 = "FAC-F-123456"
result_fac2 = validar_factura(fac_code2)
mostrar_resultado({
    "codigo": fac_code2,
    "tipo": "factura",
    "valido": result_fac2["valido"],
    "detalles": result_fac2
})

PRUEBA DE FUNCIONES INDIVIDUALES

-- Productos --
✓ TEC-0001-MX                    | Tipo: producto    
   └── valido: True, categoria: TEC, numero: 0001, pais: MX
✗ tec-0001-MX                    | Tipo: producto    

-- Envíos --
✗ ENV-2024-03-15-001234          | Tipo: envio       
✗ ENV-2024-13-15-001234          | Tipo: envio       

-- Empleados --
✗ EMP-VEN-1234                   | Tipo: empleado    
✗ EMP-VEN-0123                   | Tipo: empleado    

-- Facturas --
✓ FAC-A-123456                   | Tipo: factura     
   └── valido: True, serie: A, numero: 123456
✗ FAC-F-123456                   | Tipo: factura     


In [99]:
# Prueba validador universal

print("\nPRUEBA DEL VALIDADOR UNIVERSAL")
print("=" * 50)

for codigo in CODIGOS_PRUEBA:
    # Get the result from the universal validator
    res = validar_codigo(codigo)
    # If the 'tipo' is None (meaning no specific type was matched), set it to 'desconocido'
    if res['tipo'] is None:
        res['tipo'] = 'desconocido'
    mostrar_resultado(res)


PRUEBA DEL VALIDADOR UNIVERSAL
✓ TEC-0001-MX                    | Tipo: producto    
   └── valido: True, categoria: TEC, numero: 0001, pais: MX
✓ ALI-9999-US                    | Tipo: producto    
   └── valido: True, categoria: ALI, numero: 9999, pais: US
✓ ROB-1234-CA                    | Tipo: producto    
   └── valido: True, categoria: ROB, numero: 1234, pais: CA
✗ tec-0001-MX                    | Tipo: desconocido 
✗ TEC-001-MX                     | Tipo: desconocido 
✗ TECH-0001-MX                   | Tipo: desconocido 
✗ ENV-2024-03-15-001234          | Tipo: envio       
✗ ENV-2025-12-01-999999          | Tipo: envio       
✗ ENV-2019-03-15-001234          | Tipo: envio       
✗ ENV-2024-13-15-001234          | Tipo: envio       
✗ ENV-2024-03-32-001234          | Tipo: envio       
✗ EMP-VEN-1234                   | Tipo: empleado    
✗ EMP-TEC-9999                   | Tipo: empleado    
✗ EMP-ADM-1000                   | Tipo: empleado    
✗ EMP-VEN-0123                  

In [100]:
# Prueba procesamiento por lotes

print("\nPRUEBA DEL PROCESAMIENTO DE LOTES")
reporte = procesar_lote(CODIGOS_PRUEBA)
mostrar_reporte(reporte)


PRUEBA DEL PROCESAMIENTO DE LOTES
                 REPORTE DE VALIDACIÓN

Total procesados: 25
Válidos: 6 (24.0%)
Inválidos: 19 (76.0%)

Desglose por tipo:
----------------------------------------
  Producto    :   3/3   (100% válidos)
  Envio       :   0/5   (0% válidos)
  Empleado    :   0/6   (0% válidos)
  Factura     :   3/6   (50% válidos)
  Desconocido :   0/5   (0% válidos)



PATRONES REGEX

1. validar_producto:  r'^([A-Z]{3})-(\d{4})-([A-Z]{2})$'

* ^ : inicio de la cadena
* ([A-Z]{3}) : Exactamente 3 letras mayusculas (categoria)
* (\d{4}) : exactamente 4 dígitos (número del producto).
* ([A-Z]{2}) : exactamente 2 letras mayúsculas (código país).
* $ : fin de la cadena.


2. validar_envio : r'^ENV-(\d{4})-(\d{2})-(\d{2})-(\d{6})$'

* ^ENV- : debe empezar con "ENV-".
* (\d{4}) : año, 4 dígitos.
* -(\d{2}) : mes, 2 dígitos.
* -(\d{2}) : día, 2 dígitos.
* -(\d{6}) : secuencial, 6 dígitos.
* $ : fin de cadena.


3. validar_empleado: r'^EMP-([A-Z]{3})-(\d{4})$'

* ^EMP- : debe empezar con "EMP-".
* ([A-Z]{3}) : departamento, 3 letras mayúsculas.
* -(\d{4}) : número de empleado, 4 dígitos.
* $ : fin de cadena.


4. validar_factura : r'^FAC-([A-Z])-([0-9]{6})$'

* ^FAC- : debe empezar con "FAC-".
* ([A-Z]) : serie, una sola letra mayúscula (A–E).
* -([0-9]{6}) : número de factura, 6 dígitos.
* $ : fin de cadena.

